In [31]:
from dotenv import load_dotenv
load_dotenv()

True

In [32]:
from langgraph.graph import StateGraph,START,END
from langgraph.checkpoint.memory import InMemorySaver
from langchain_groq import ChatGroq
from pydantic import BaseModel,Field
from typing import Annotated
from langgraph.graph.message import add_messages


In [33]:
class graphstate(BaseModel):
    message:Annotated[list,add_messages]

llm=ChatGroq(model="openai/gpt-oss-20b")

In [34]:
def chat_node(state:graphstate)->graphstate:
    res=llm.invoke(state.message)
    state.message=[res]
    return state

In [35]:
graph=StateGraph(graphstate)

In [36]:
graph.add_node("llm",chat_node)

In [37]:
graph.add_edge(START,"llm")

In [38]:
graph.add_edge("llm",END)

In [39]:
final=graph.compile(checkpointer=InMemorySaver())

In [45]:
res=final.invoke({"message":[{"role":"user","content":"what are its applications?"}]},
             {"configurable":{"thread_id":"1"}})

In [46]:
res["message"][-1].content

'## RAG in the Real World  \nBelow are the most common ways companies and developers are using Retrieval‑Augmented Generation today—grouped by domain and with concrete examples.\n\n| Domain | Typical Use Case | Why RAG Helps | Example Tools / Platforms |\n|--------|------------------|---------------|---------------------------|\n| **Customer Support / Help‑Desk** | *Answer a user’s question using the latest product docs.* | Keeps answers fresh (e.g., new API changes) and reduces support tickets. | Chatbot on Zendesk + LangChain RAG, Microsoft Bot Framework + Azure Cognitive Search |\n| **Enterprise Knowledge Management** | *Internal “Ask an Expert” portal that pulls from policy docs, SOPs, code repos.* | Employees get accurate, up‑to‑date guidance without hunting through intranet. | Confluence + ElasticSearch + GPT‑4 RAG, SharePoint + Azure OpenAI |\n| **Legal & Compliance** | *Generate legal briefs that reference current statutes and case law.* | Grounded responses reduce risk of citi